# 04 — SageMaker Pipeline (DAG): Preprocess → Train → Evaluate → Register

This notebook builds and runs a **SageMaker Pipeline** that:

1. Preprocesses curated CSV into supervised features and train/val/test splits
2. Trains a scikit-learn model in a SageMaker Training Job
3. Evaluates the model and writes `evaluation.json`
4. **Conditionally registers** the model to **SageMaker Model Registry**
   - Pass = registers model
   - Fail = pipeline fails (great for demoing a failed DAG)

This satisfies:
- **Pipeline/DAG** requirement
- **Model Registry** requirement


In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

import time
import boto3
import sagemaker

from pipelines.buoycast_pipeline import get_pipeline

In [ ]:
# Load from previous notebook
%store -r bucket
%store -r region
%store -r CURATED_PREFIX

print("Bucket:", bucket)
print("Region:", region)
print("CURATED_PREFIX:", CURATED_PREFIX)

In [ ]:
# SageMaker context
sess = sagemaker.Session()
role = sagemaker.get_execution_role()

curated_s3_prefix = f"s3://{bucket}/{CURATED_PREFIX}/"
artifacts_prefix = f"s3://{bucket}/buoycast/artifacts"

PIPELINE_NAME = "buoycast-train-register"
MODEL_PACKAGE_GROUP = "buoycast-wave-models"

# RunId makes it easy to find artifacts in S3 later
RUN_ID = time.strftime("%Y%m%d-%H%M%S")

print("Role:", role)
print("Curated:", curated_s3_prefix)
print("Artifacts:", artifacts_prefix)
print("RunId:", RUN_ID)

In [ ]:
# Create / update pipeline definition
pipeline = get_pipeline(
    region=region,
    role=role,
    default_bucket=bucket,
    pipeline_name=PIPELINE_NAME,
    base_job_prefix="buoycast",
)

pipeline.upsert(role_arn=role)
print("Upserted pipeline:", PIPELINE_NAME)

In [ ]:
# ---- Start a SUCCESSFUL run ----
# Increase/decrease threshold to control pass/fail.
# If you want a guaranteed FAIL run, set RmseHsThreshold to something tiny like 0.0001.
execution = pipeline.start(
    parameters={
        "CuratedS3Prefix": curated_s3_prefix,
        "ArtifactsS3Prefix": artifacts_prefix,
        "RunId": RUN_ID,
        "ModelPackageGroupName": MODEL_PACKAGE_GROUP,
        "ModelApprovalStatus": "Approved",
        "RmseHsThreshold": 0.5,
        "ProcessingInstanceType": "ml.m5.xlarge",
        "TrainingInstanceType": "ml.m5.xlarge",
    }
)

print("PipelineExecutionArn:", execution.arn)

In [ ]:
# Monitor execution (optional: blocks until completion)
execution.describe()

In [ ]:
# If you want to WAIT until the pipeline finishes, run this cell:
# execution.wait()

# Then list the steps + statuses:
execution.list_steps()

In [ ]:
pipeline_execution_arn = execution.arn

%store PIPELINE_NAME
%store MODEL_PACKAGE_GROUP
%store RUN_ID
%store pipeline_execution_arn


## Demoing a failed DAG

To show a failed pipeline execution in your video demo:

- Re-run the `pipeline.start(...)` cell, but set `RmseHsThreshold` to a value that the model cannot meet (e.g., `0.0001`).
- The pipeline will hit the **FailIfBadMetrics** step and stop.
